# Notebook 02 — Build User Journeys

## 🎯 Objective
Transform impression-level logs into **ordered user journeys** (sequences of touchpoints per user leading up to a conversion or not).  
These journey tables are the foundation for multi-touch attribution (MTA), path analytics, and downstream modeling.

---

## 🧱 What this notebook does

1. **Project path setup**
   - Adds the project root to `PYTHONPATH` so we can import custom modules from `src/`.
   - Verifies imports:
     - `src.journeys.build.load_impressions_from_parquet`
     - `src.journeys.build.load_impressions_from_tsv`
     - `src.journeys.build.build_and_save`

2. **Load impressions (1 row = 1 impression)**
   - Preferred: `../data/interim/pcb_dataset.parquet` (fast, memory-efficient).
   - Alternative: raw TSV loader (slower, requires dtypes).

3. **Build journeys (via src/journeys/build.py)**
   - Calls `build_and_save(impressions=..., out_all=..., out_conv=...)`from src/journeys/build.py.
   - Produces two Parquet datasets:
     - **All journeys**: `../data/processed/journeys_all.parquet`
     - **Converted journeys only**: `../data/processed/journeys_converted.parquet`

4. **Sanity checks**
   - Confirms overall shapes and positive labels.
   - Prints value counts for `conversion` in both outputs.
   - Displays example rows to verify sequence fields.

---

## 🧩 Under the hood: src/journeys/build.py

### Key functions

- `build_journeys_from_impressions(df: pd.DataFrame) -> pd.DataFrame`
Stitches ordered impressions into journey segments
(one row per conversion segment + optional post-conversion tail).

- `build_and_save(impressions, out_all, out_conv)`
Convenience wrapper: builds journeys and writes both
journeys_all.parquet and journeys_converted.parquet.

- `load_impressions_from_parquet(path) / load_impressions_from_tsv(path, dtypes)`
Lightweight I/O helpers for loading impression logs.


### Required impression columns: 
timestamp, uid, campaign, conversion_id, conversion_timestamp, click, cost, time_since_last_click

---

## 📦 Inputs & Outputs

### Input
- `../data/interim/pcb_dataset.parquet` (from Notebook 01)  
  Columns (key): `uid, campaign, timestamp, click, cost, conversion, conversion_id, conversion_timestamp, time_since_last_click, ...`

### Outputs
- `../data/processed/journeys_all.parquet`
- `../data/processed/journeys_converted.parquet`

Each row in these files represents **one journey** (a user path consisting of multiple touchpoints).

---

## 🧾 Journey schema (columns)

| Column | Type | Description |
|---|---|---|
| `uid` | int/str | User identifier for the journey |
| `conversion` | int (0/1) | Whether the journey ended in a conversion |
| `conversion_id` | int | Conversion event id (or `-1` if none) |
| `conv_time` | int | Conversion timestamp (or `-1` if none) |
| `length` | int | Number of touchpoints in the journey |
| `campaign_seq` | list[int] | Ordered campaign IDs across the path |
| `click_seq` | list[int] | Ordered click flags per touchpoint (0/1) |
| `cost_seq` | list[float] | Ordered cost per touchpoint |
| `time_since_last_click_seq` | list[float] | Ordered intervals; `-1` when unknown |
| `first_campaign_id` | int | First touchpoint campaign in journey |
| `last_campaign_id` | int | Last touchpoint campaign in journey |
| `total_cost` | float | Sum of costs across touchpoints |
| `num_clicks` | int | Sum of clicks across touchpoints |

> Note: `-1` is a placeholder for "no event" or "unknown" (consistent with raw data).

---

## 📊 Results from this run

- **All journeys**: `6,814,951` rows  
  - Positives (`conversion=1`): `806,196`
  - Journey-level conversion rate: **≈ 11.83%**
- **Converted journeys only**: `806,196` rows (all `conversion=1`)

> There are ~6.1M unique users in the raw data and ~6.8M journeys, implying some users have multiple journeys (e.g., multiple sessions or windows).

---

## 🧠 Why journeys matter (Business & Analytics)

- **True multi-touch view**: Campaigns rarely work in isolation. Journeys expose **order and combination effects** (e.g., *Awareness → Consideration → Retargeting*).
- **Attribution modeling**: This structure enables:
  - **Heuristic** rules (first-touch, last-touch, linear)
  - **Data-driven** methods (Shapley value attribution)
  - **Uplift/incrementality-aware** modeling (when experiments or priors exist)
- **Optimization levers**:
  - Identify **assist** campaigns that rarely close but increase downstream conversion.
  - Tune **frequency capping** and **sequencing** (e.g., avoid long, costly paths that don’t convert).
  - Reallocate budget from high-cost/low-assist paths to **efficient** paths.
- **Segmentation**:
  - Cluster journeys by **length**, **cost**, **click density**, **campaign mixes**, **time gaps**.
  - Design creatives and sequences per **journey archetype**.

---

## 🔌 How this fits the pipeline
1. **Notebook 01 — Data Preprocessing & EDA** 
   Prepare impression-level data, KPIs, and Parquet artifacts.
2. **Notebook 02 — Build User Journeys** (This) 
   Convert impressions → ordered user journeys with sequences.
3. **Notebook 03 — Attribution & Modeling (3a–3c)**  
   - **03a — Heuristic Attribution:** First/Last/Linear/Time-decay over journeys.  
   - **03b — Journey Conversion Modeling:** Logistic Regression & XGBoost with calibration + SHAP.  
   - **03c — Shapley Attribution (Monte Carlo):** Model-based credit per campaign.
4. **Notebook 04 — Attribution Rollups, ROI & Rule-vs-Model Comparison**  
   Consolidate attributions, compute ROI, compare methods.
5. **Notebook 05 — Customer Segmentation & Audience Exports**  
   KMeans segmentation, campaign-mix profiles, and BI exports.
6. **Notebook 06 — Budget Reallocation Simulator (Upcoming)**  
   What-if scenarios using attributed ROI and constraints.

---

In [3]:
import sys, os
PROJECT_ROOT = os.path.abspath("..")  
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PYTHONPATH includes:", PROJECT_ROOT in sys.path)


PYTHONPATH includes: True


In [9]:
import pandas as pd
from src.journeys.build import load_impressions_from_parquet, load_impressions_from_tsv

impr = load_impressions_from_parquet("../data/interim/pcb_dataset.parquet")

print(impr.shape, impr.columns.tolist()[:18])

(16468027, 22) ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5']


In [11]:
from src.journeys.build import build_and_save

J_all, J_conv = build_and_save(
    impressions=impr,
    out_all="../data/processed/journeys_all.parquet",
    out_conv="../data/processed/journeys_converted.parquet"
)

print("Journeys (ALL):", J_all.shape, "pos=", int(J_all['conversion'].sum()))
print("Journeys (CONVERTED ONLY):", J_conv.shape)


Journeys (ALL): (6814951, 13) pos= 806196
Journeys (CONVERTED ONLY): (806196, 13)


In [12]:
print("ALL -> conversion value counts")
print(J_all["conversion"].value_counts(dropna=False))

print("\nConverted journeys stats (should all be 1):")
print(J_conv["conversion"].value_counts(dropna=False))

# Sanity: example rows
display(J_all.head(3))
display(J_conv.head(3))

ALL -> conversion value counts
conversion
0    6008755
1     806196
Name: count, dtype: int64

Converted journeys stats (should all be 1):
conversion
1    806196
Name: count, dtype: int64


,uid,conversion,conversion_id,conv_time,length,campaign_seq,click_seq,cost_seq,time_since_last_click_seq,first_campaign_id,last_campaign_id,total_cost,num_clicks
0,10000148,0,-1,-1,16,"[13166099, 31772643, 31772643, 31772643, 31772...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.0001880141207948327, 9.999999747378752e-06,...","[220193.0, 269760.0, 666201.0, 666771.0, 70166...",13166099,13166099,0.001507,0
1,10001508,0,-1,-1,27,"[20916101, 20916101, 20916101, 20916101, 20916...","[0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, ...","[2.388120992691256e-05, 0.00011574133532121778...","[330658.0, 330719.0, 1523.0, 2387.0, 56549.0, ...",20916101,15654890,0.002062,16
2,1000222,0,-1,-1,7,"[26321366, 26321366, 26321366, 26321366, 26321...","[1, 0, 0, 1, 1, 0, 1]","[8.647569484310225e-05, 9.956641588360071e-05,...","[134205.0, 208040.0, 462542.0, 606775.0, 241.0...",26321366,26321366,0.000852,4


,uid,conversion,conversion_id,conv_time,length,campaign_seq,click_seq,cost_seq,time_since_last_click_seq,first_campaign_id,last_campaign_id,total_cost,num_clicks
22,10026553,1,2121096,1769555,5,"[18436526, 12121528, 18436526, 18436526, 12121...","[1, 0, 0, 0, 1]","[0.0001444051304133609, 0.00015126430662348866...","[-1.0, 930611.0, 84747.0, 1063760.0, 2316958.0]",18436526,12121528,0.001017,2
23,10026553,1,21912265,2205708,1,[12121528],[1],[0.0039362115785479546],[749727.0],12121528,12121528,0.003936,1
25,1002720,1,9890254,2699469,5,"[15491650, 9100689, 15491650, 15491650, 14121532]","[0, 0, 1, 1, 1]","[0.0001476796460337937, 0.00013106329424772412...","[-1.0, -1.0, -1.0, 615997.0, -1.0]",15491650,14121532,0.000816,3
